[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-USERNAME/AI-in-healthcare-book/blob/main/notebooks/chapter_05/notebook_5_2_roc_pr_curves.ipynb)

*Click the badge above to open this notebook in Google Colab (no setup required)*

---


# 5.2 ROC and Precision-Recall Curves

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Construct and interpret ROC curves** and understand what AUROC represents
2. **Construct and interpret Precision-Recall (PR) curves** and AUPRC
3. **Choose between ROC and PR curves** based on class balance and clinical context
4. **Select optimal operating points** based on clinical costs and constraints
5. **Compare multiple models** using curve-based metrics
6. **Understand common pitfalls** in ROC/PR interpretation

---

## Clinical Context: Why Curves Matter

**Journey Connections:**
- **Marcus (Sepsis)**: High sensitivity needed → understanding where on ROC curve to operate
- **Yuki (AFib)**: Rare condition (0.3% prevalence) → PR curves more informative than ROC
- **Jamal (Lung Nodules)**: Screening context → need to see full sensitivity-specificity trade-off

**Clinical Reality:**
- A single metric (AUROC = 0.85) doesn't tell you:
  - What sensitivity/specificity you'll actually get in practice
  - How performance changes with different decision thresholds
  - Whether the model is useful for YOUR clinical scenario
- ROC curves show ALL possible operating points
- PR curves reveal performance in imbalanced settings

**Example:**
- Model A: AUROC = 0.90 but unusable sensitivity at acceptable FPR
- Model B: AUROC = 0.85 but better in the operating range you need
- **Curves reveal this; single metrics don't**

---

## Setup

In [ ]:
# Standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    roc_curve, auc, roc_auc_score,
    precision_recall_curve, average_precision_score,
    confusion_matrix, classification_report
)
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 10

# Set random seed for reproducibility
np.random.seed(42)

print("Setup complete!")

---

## Part 1: Understanding ROC Curves

### What is an ROC Curve?

**ROC (Receiver Operating Characteristic) Curve:**
- Plots **True Positive Rate (TPR/Sensitivity)** vs **False Positive Rate (FPR)**
- Shows performance across ALL possible decision thresholds
- Each point = (FPR, TPR) at one threshold

**Key Concepts:**
- TPR = TP / (TP + FN) = Sensitivity = Recall
- FPR = FP / (FP + TN) = 1 - Specificity
- **AUROC (Area Under ROC)**: Single metric summarizing curve (0.5 = random, 1.0 = perfect)

**Interpretation:**
- **Top-left corner** = ideal (TPR=1, FPR=0)
- **Diagonal line** = random guessing (AUROC = 0.5)
- **Above diagonal** = better than random
- **Curve shape** matters more than AUROC for clinical use

### 1.1 Generate Synthetic Clinical Data

In [ ]:
def generate_sepsis_data(n_samples=2000, prevalence=0.15, difficulty='moderate'):
    """
    Generate synthetic sepsis prediction data.

    Parameters:
    -----------
    n_samples : int
        Number of patients
    prevalence : float
        Proportion with sepsis (0.15 = 15%)
    difficulty : str
        'easy', 'moderate', 'hard' - controls class separability

    Returns:
    --------
    X : array
        Features (vitals, labs)
    y : array
        Labels (1 = sepsis, 0 = no sepsis)
    feature_names : list
        Feature names
    """
    # Map difficulty to class separation
    sep_map = {'easy': 3.0, 'moderate': 1.5, 'hard': 0.5}
    class_sep = sep_map.get(difficulty, 1.5)

    # Generate data
    X, y = make_classification(
        n_samples=n_samples,
        n_features=8,
        n_informative=6,
        n_redundant=2,
        n_classes=2,
        weights=[1-prevalence, prevalence],
        class_sep=class_sep,
        flip_y=0.05,  # 5% label noise
        random_state=42
    )

    # Transform to realistic clinical ranges
    # Feature 0-2: Vitals (HR, RR, Temp)
    X[:, 0] = 70 + X[:, 0] * 20 + y * 15  # HR: 50-110, higher in sepsis
    X[:, 1] = 16 + X[:, 1] * 4 + y * 6    # RR: 12-26, higher in sepsis
    X[:, 2] = 37 + X[:, 2] * 1.5 + y * 1  # Temp: 35-40, higher in sepsis

    # Feature 3-5: Labs (WBC, Lactate, Creatinine)
    X[:, 3] = 8 + X[:, 3] * 3 + y * 5     # WBC: 5-20, higher in sepsis
    X[:, 4] = 1.5 + X[:, 4] * 1 + y * 2   # Lactate: 0.5-5, higher in sepsis
    X[:, 5] = 0.9 + X[:, 5] * 0.3 + y * 0.5  # Creatinine: 0.6-2.0

    # Feature 6-7: Patient factors (age, comorbidities)
    X[:, 6] = 50 + X[:, 6] * 20  # Age: 30-70
    X[:, 7] = np.abs(X[:, 7])    # Comorbidity score: 0-3

    feature_names = [
        'heart_rate', 'resp_rate', 'temperature',
        'wbc', 'lactate', 'creatinine',
        'age', 'comorbidity_score'
    ]

    return X, y, feature_names

# Generate data
X, y, feature_names = generate_sepsis_data(n_samples=2000, prevalence=0.15)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

# Standardize
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Data shape: {X.shape}")
print(f"Sepsis prevalence: {y.mean():.1%}")
print(f"Train set: {len(y_train)} patients")
print(f"Test set: {len(y_test)} patients")

### 1.2 Train a Model and Generate Predictions

In [ ]:
# Train logistic regression
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train_scaled, y_train)

# Get predicted probabilities (NOT binary predictions)
# This is CRITICAL for ROC/PR curves - we need probabilities!
y_proba = lr_model.predict_proba(X_test_scaled)[:, 1]  # Probability of class 1 (sepsis)

print("Predicted probabilities (first 10 patients):")
print(y_proba[:10])
print(f"\nProbability range: [{y_proba.min():.3f}, {y_proba.max():.3f}]")
print(f"Mean probability: {y_proba.mean():.3f}")

### 1.3 Construct ROC Curve Manually

**Understanding how ROC is built:**
1. Sort patients by predicted probability (high → low)
2. For each unique probability threshold:
   - Classify as positive if prob ≥ threshold
   - Calculate TPR and FPR at this threshold
3. Plot (FPR, TPR) points

In [ ]:
def manual_roc_curve(y_true, y_proba, n_thresholds=100):
    """
    Manually construct ROC curve to understand the mechanics.

    Parameters:
    -----------
    y_true : array
        True labels
    y_proba : array
        Predicted probabilities
    n_thresholds : int
        Number of threshold points to evaluate

    Returns:
    --------
    fpr : array
        False positive rates
    tpr : array
        True positive rates
    thresholds : array
        Decision thresholds
    """
    # Define thresholds from 0 to 1
    thresholds = np.linspace(0, 1, n_thresholds)

    fpr_list = []
    tpr_list = []

    for threshold in thresholds:
        # Make binary predictions at this threshold
        y_pred = (y_proba >= threshold).astype(int)

        # Calculate confusion matrix
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

        # Calculate TPR and FPR
        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0

        fpr_list.append(fpr)
        tpr_list.append(tpr)

    return np.array(fpr_list), np.array(tpr_list), thresholds

# Build ROC curve manually
fpr_manual, tpr_manual, thresholds_manual = manual_roc_curve(y_test, y_proba)

# Show a few points
print("ROC Curve Points (selected thresholds):")
print("Threshold | FPR    | TPR")
print("-" * 30)
for i in [0, 25, 50, 75, 99]:
    print(f"{thresholds_manual[i]:.2f}      | {fpr_manual[i]:.3f}  | {tpr_manual[i]:.3f}")

print("\nKey insights:")
print(f"- At threshold 0.00 → FPR=1.0, TPR=1.0 (classify everyone as positive)")
print(f"- At threshold 1.00 → FPR=0.0, TPR=0.0 (classify everyone as negative)")
print(f"- At threshold 0.50 → FPR={fpr_manual[50]:.3f}, TPR={tpr_manual[50]:.3f}")

### 1.4 Plot ROC Curve

In [ ]:
# Use sklearn's roc_curve for comparison
fpr_sklearn, tpr_sklearn, thresholds_sklearn = roc_curve(y_test, y_proba)
auroc = auc(fpr_sklearn, tpr_sklearn)

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Plot 1: ROC curve
ax1.plot(fpr_sklearn, tpr_sklearn, 'b-', linewidth=2, label=f'Model (AUROC = {auroc:.3f})')
ax1.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier (AUROC = 0.500)')

# Mark some operating points
# Find threshold closest to 0.5
idx_05 = np.argmin(np.abs(thresholds_sklearn - 0.5))
ax1.plot(fpr_sklearn[idx_05], tpr_sklearn[idx_05], 'ro', markersize=10,
         label=f'Threshold=0.5: FPR={fpr_sklearn[idx_05]:.3f}, TPR={tpr_sklearn[idx_05]:.3f}')

# Find point with TPR ≈ 0.90
idx_90 = np.argmin(np.abs(tpr_sklearn - 0.90))
ax1.plot(fpr_sklearn[idx_90], tpr_sklearn[idx_90], 'gs', markersize=10,
         label=f'90% Sensitivity: FPR={fpr_sklearn[idx_90]:.3f}')

ax1.set_xlabel('False Positive Rate (1 - Specificity)', fontsize=12)
ax1.set_ylabel('True Positive Rate (Sensitivity)', fontsize=12)
ax1.set_title('ROC Curve: Sepsis Prediction Model', fontsize=14, fontweight='bold')
ax1.legend(loc='lower right')
ax1.grid(True, alpha=0.3)
ax1.set_xlim([-0.02, 1.02])
ax1.set_ylim([-0.02, 1.02])

# Plot 2: Threshold vs Metrics
ax2.plot(thresholds_sklearn, tpr_sklearn, 'b-', linewidth=2, label='TPR (Sensitivity)')
ax2.plot(thresholds_sklearn, 1 - fpr_sklearn, 'r-', linewidth=2, label='TNR (Specificity)')
ax2.axvline(0.5, color='gray', linestyle='--', alpha=0.5, label='Default threshold')
ax2.set_xlabel('Classification Threshold', fontsize=12)
ax2.set_ylabel('Rate', fontsize=12)
ax2.set_title('Sensitivity-Specificity Trade-off', fontsize=14, fontweight='bold')
ax2.legend(loc='best')
ax2.grid(True, alpha=0.3)
ax2.set_xlim([0, 1])
ax2.set_ylim([0, 1.02])

plt.tight_layout()
plt.show()

print("\nROC Curve Interpretation:")
print(f"AUROC = {auroc:.3f}")
print(f"\nAt 90% sensitivity (catching 90% of sepsis cases):")
print(f"  - FPR = {fpr_sklearn[idx_90]:.3f} ({fpr_sklearn[idx_90]*100:.1f}% false alarm rate)")
print(f"  - Specificity = {1-fpr_sklearn[idx_90]:.3f}")
print(f"\nAt default threshold (0.5):")
print(f"  - Sensitivity = {tpr_sklearn[idx_05]:.3f}")
print(f"  - Specificity = {1-fpr_sklearn[idx_05]:.3f}")

**Clinical Interpretation:**

The ROC curve shows:
1. **AUROC = ~0.85**: Model has good discrimination
2. **Trade-off is real**: To catch 90% of sepsis cases, we accept ~20% false positive rate
3. **Threshold matters**: Default 0.5 might not be optimal for clinical use

**Marcus's Story (Journey 1):**
- Hospital chose high sensitivity (90%) to minimize missed sepsis
- Accepted higher FPR (more false alarms)
- ROC curve helps visualize this choice

---

## Part 2: Precision-Recall Curves

### Why PR Curves?

**ROC curves can be misleading for imbalanced datasets!**

**Example:**
- AFib prevalence = 0.3% (Yuki's story)
- 1000 patients: 3 with AFib, 997 without
- Model with 90% sensitivity, 95% specificity:
  - TP = 2.7 ≈ 3
  - FP = 49.85 ≈ 50
  - **FPR = 50/997 = 0.05** (looks great!)
  - **Precision = 3/(3+50) = 0.057** (terrible!)

**Precision-Recall (PR) Curve:**
- Plots **Precision (PPV)** vs **Recall (Sensitivity/TPR)**
- Precision = TP / (TP + FP) - "Of all positive predictions, how many were correct?"
- Recall = TP / (TP + FN) - "Of all actual positives, how many did we catch?"
- **More informative for rare diseases** because precision focuses on positive predictions

### 2.1 Generate Imbalanced Dataset (AFib Screening)

In [ ]:
def generate_afib_data(n_samples=10000, prevalence=0.003):
    """
    Generate synthetic AFib detection data (highly imbalanced).
    Prevalence = 0.3% (similar to Yuki's journey)
    """
    X, y = make_classification(
        n_samples=n_samples,
        n_features=10,
        n_informative=7,
        n_redundant=3,
        n_classes=2,
        weights=[1-prevalence, prevalence],
        class_sep=1.2,
        flip_y=0.02,
        random_state=42
    )

    # Transform to ECG-like features
    X[:, 0] = 60 + X[:, 0] * 15 + y * 20  # Heart rate variability
    X[:, 1] = 0.15 + X[:, 1] * 0.05 + y * 0.03  # RR interval variance
    X[:, 2] = 40 + X[:, 2] * 10 + y * 5  # PR interval
    X[:, 3] = 0.08 + X[:, 3] * 0.02 + y * 0.01  # QRS duration

    return X, y

# Generate AFib data
X_afib, y_afib = generate_afib_data(n_samples=10000, prevalence=0.003)

# Split
X_train_afib, X_test_afib, y_train_afib, y_test_afib = train_test_split(
    X_afib, y_afib, test_size=0.3, stratify=y_afib, random_state=42
)

# Standardize
scaler_afib = StandardScaler()
X_train_afib_scaled = scaler_afib.fit_transform(X_train_afib)
X_test_afib_scaled = scaler_afib.transform(X_test_afib)

print(f"AFib Dataset:")
print(f"Total samples: {len(y_afib)}")
print(f"AFib cases: {y_afib.sum()} ({y_afib.mean()*100:.2f}%)")
print(f"Class imbalance ratio: {(1-y_afib.mean())/y_afib.mean():.0f}:1")
print(f"\nTest set AFib cases: {y_test_afib.sum()}")

### 2.2 Train Model on Imbalanced Data

In [ ]:
# Train logistic regression on AFib data
lr_afib = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')
lr_afib.fit(X_train_afib_scaled, y_train_afib)

# Get probabilities
y_proba_afib = lr_afib.predict_proba(X_test_afib_scaled)[:, 1]

print(f"Predicted probability statistics:")
print(f"Mean: {y_proba_afib.mean():.4f}")
print(f"Median: {np.median(y_proba_afib):.4f}")
print(f"Max: {y_proba_afib.max():.4f}")
print(f"\nNote: Most probabilities are very low (rare disease)")

### 2.3 Compare ROC vs PR Curves on Imbalanced Data

In [ ]:
# Calculate ROC
fpr_afib, tpr_afib, _ = roc_curve(y_test_afib, y_proba_afib)
auroc_afib = auc(fpr_afib, tpr_afib)

# Calculate PR
precision_afib, recall_afib, _ = precision_recall_curve(y_test_afib, y_proba_afib)
auprc_afib = auc(recall_afib, precision_afib)
avg_precision_afib = average_precision_score(y_test_afib, y_proba_afib)

# Plot both
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# ROC Curve
ax1.plot(fpr_afib, tpr_afib, 'b-', linewidth=2, label=f'Model (AUROC = {auroc_afib:.3f})')
ax1.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
ax1.set_xlabel('False Positive Rate', fontsize=12)
ax1.set_ylabel('True Positive Rate (Recall)', fontsize=12)
ax1.set_title('ROC Curve: AFib Detection\n(Looks Great!)', fontsize=14, fontweight='bold')
ax1.legend(loc='lower right')
ax1.grid(True, alpha=0.3)

# PR Curve
baseline_precision = y_test_afib.mean()  # Prevalence
ax2.plot(recall_afib, precision_afib, 'r-', linewidth=2,
         label=f'Model (AUPRC = {avg_precision_afib:.3f})')
ax2.axhline(baseline_precision, color='k', linestyle='--', linewidth=1,
            label=f'Random (baseline = {baseline_precision:.3f})')
ax2.set_xlabel('Recall (Sensitivity)', fontsize=12)
ax2.set_ylabel('Precision (PPV)', fontsize=12)
ax2.set_title('PR Curve: AFib Detection\n(Shows True Challenge!)', fontsize=14, fontweight='bold')
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.3)
ax2.set_xlim([0, 1])
ax2.set_ylim([0, 1])

plt.tight_layout()
plt.show()

print("\nComparison: ROC vs PR for Imbalanced Data")
print("=" * 60)
print(f"Prevalence: {y_test_afib.mean()*100:.2f}% (very rare!)\n")
print(f"ROC Curve:")
print(f"  AUROC = {auroc_afib:.4f} <- Looks excellent!")
print(f"\nPR Curve:")
print(f"  AUPRC = {avg_precision_afib:.4f} <- Shows true challenge!")
print(f"  Baseline (random) = {baseline_precision:.4f}")
print(f"\nWarning: ROC curve is MISLEADING here because:")
print(f"  - FPR uses denominator of negatives (997/1000)")
print(f"  - Even 50 false positives -> FPR = 0.05 (looks great)")
print(f"  - But Precision = 3/(3+50) = 0.057 (terrible!)")
print(f"\nPR curve reveals:")
print(f"  - Precision directly shows PPV (what clinicians care about)")
print(f"  - High recall requires sacrificing precision (low PPV)")
print(f"  - Model struggles with rare positive class")

**Yuki's Story Insight:**

With 0.3% AFib prevalence:
- **ROC curve**: AUROC = 0.96 → "Excellent model!"
- **PR curve**: AUPRC = 0.15 → "95% of positive predictions are false alarms"

**The ROC curve hides the problem** because:
- True negatives dominate (997 out of 1000)
- FPR stays low even with many false positives
- PR curve forces us to confront: "How many alarms are real?"

---

## Part 3: Comparing Multiple Models

In [ ]:
# Train three models
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, max_depth=5),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42, max_depth=3)
}

results = {}

for name, model in models.items():
    # Train
    model.fit(X_train_scaled, y_train)

    # Predict probabilities
    y_proba_model = model.predict_proba(X_test_scaled)[:, 1]

    # Calculate curves
    fpr_model, tpr_model, _ = roc_curve(y_test, y_proba_model)
    auroc_model = auc(fpr_model, tpr_model)

    precision_model, recall_model, _ = precision_recall_curve(y_test, y_proba_model)
    auprc_model = average_precision_score(y_test, y_proba_model)

    results[name] = {
        'fpr': fpr_model,
        'tpr': tpr_model,
        'auroc': auroc_model,
        'precision': precision_model,
        'recall': recall_model,
        'auprc': auprc_model
    }

    print(f"{name}:")
    print(f"  AUROC: {auroc_model:.4f}")
    print(f"  AUPRC: {auprc_model:.4f}")
    print()

# Plot comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

colors = ['blue', 'green', 'red']
linestyles = ['-', '--', '-.']

# ROC curves
for (name, res), color, ls in zip(results.items(), colors, linestyles):
    ax1.plot(res['fpr'], res['tpr'], color=color, linestyle=ls, linewidth=2,
             label=f"{name} (AUROC={res['auroc']:.3f})")

ax1.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.3, label='Random')
ax1.set_xlabel('False Positive Rate', fontsize=12)
ax1.set_ylabel('True Positive Rate', fontsize=12)
ax1.set_title('ROC Curves: Model Comparison', fontsize=14, fontweight='bold')
ax1.legend(loc='lower right')
ax1.grid(True, alpha=0.3)

# PR curves
baseline = y_test.mean()
for (name, res), color, ls in zip(results.items(), colors, linestyles):
    ax2.plot(res['recall'], res['precision'], color=color, linestyle=ls, linewidth=2,
             label=f"{name} (AUPRC={res['auprc']:.3f})")

ax2.axhline(baseline, color='k', linestyle='--', linewidth=1, alpha=0.3,
            label=f'Random (baseline={baseline:.3f})')
ax2.set_xlabel('Recall', fontsize=12)
ax2.set_ylabel('Precision', fontsize=12)
ax2.set_title('PR Curves: Model Comparison', fontsize=14, fontweight='bold')
ax2.legend(loc='best')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nModel Selection Insights:")
print("=" * 60)
print("All models have similar AUROC (~0.84-0.86)")
print("But look at the CURVES:")
print("  - Which model has better sensitivity at low FPR?")
print("  - Which maintains precision at high recall?")
print("  - These differences matter for clinical deployment!")
print("\nDon't just compare AUROC - inspect the full curves!")

---

## Key Takeaways

### 1. ROC Curves
- **Show**: True Positive Rate vs False Positive Rate across all thresholds
- **AUROC**: Summary metric (0.5 = random, 1.0 = perfect)
- **Use when**: Classes are balanced, both classes matter equally
- **Limitation**: Can be misleading for imbalanced datasets

### 2. Precision-Recall Curves
- **Show**: Precision (PPV) vs Recall (Sensitivity) across thresholds
- **AUPRC**: Summary metric (baseline = prevalence)
- **Use when**: Positive class is rare, FP costs are high
- **Advantage**: Reveals true challenge of rare disease detection

### 3. When to Use Which?
- **ROC**: Balanced datasets, both classes matter equally
- **PR**: Imbalanced datasets (prevalence < 10%), rare diseases
- **Both**: Always report both for screening applications!

### 4. Common Pitfalls
- Relying only on AUROC for imbalanced data
- Not inspecting curve shape in operating range
- Ignoring calibration (curves show ranking, not probability accuracy)
- Using default threshold without clinical consideration

### 5. Best Practices
- Report both ROC and PR for imbalanced data
- Inspect full curves, not just summary metrics
- Select threshold based on clinical costs (involve clinicians!)
- Compare models at specific operating points
- Check calibration separately
- Validate on external data before deployment

---

## Exercises

### Exercise 1: ROC Curve Construction
Using the sepsis dataset:
1. Calculate sensitivity and specificity at thresholds 0.3, 0.5, 0.7
2. Plot these points on an ROC curve
3. Which threshold would you choose for a screening application? Why?

### Exercise 2: PR Curves for Rare Disease
Generate a dataset with 0.1% prevalence:
1. Train a logistic regression model
2. Plot both ROC and PR curves
3. Calculate AUROC and AUPRC
4. Explain why they differ and which is more informative

### Exercise 3: Threshold Selection
For the sepsis model:
1. Find the threshold that achieves:
   - Exactly 95% sensitivity
   - Exactly 90% specificity
2. Calculate PPV at each threshold (assume 15% prevalence)
3. Which would you deploy if:
   - Missed sepsis is very costly
   - False alarms overwhelm ICU staff

### Exercise 4: Model Comparison
Train three models (Logistic Regression, Random Forest, XGBoost):
1. Plot all three ROC curves on the same plot
2. Identify which model is best at:
   - High sensitivity (> 90%)
   - Low FPR (< 10%)
   - Overall AUROC
3. Can the same model be best at all three? Why or why not?

---

**Next Notebook**: 5.3 Calibration and Clinical Utility (David's CVD Risk Journey)